In [ ]:
# =========================================
# Bank Marketing: GBDT vs MLP
# End-to-end with clean preprocessing & visuals
# =========================================

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# -----------------------------
# Utility: evaluation report
# -----------------------------
def report(name, y_true, y_pred, proba=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    ap = average_precision_score(y_true, proba) if proba is not None else float('nan')
    print(f'[{name}] Acc={acc:.4f} | P={prec:.4f} | R={rec:.4f} | F1={f1:.4f} | AUC-PR={ap:.4f}')

# 1. Data Preparation

In [ ]:
# -----------------------------
# 1a) Load dataset
# -----------------------------
df = pd.read_csv('bank-additional-full.csv', sep=';')

print("Dataset shape:", df.shape)
print("Head:\n", df.head(3))
print("\nTarget distribution:")
print(df['y'].value_counts(normalize=True))

In [ ]:
# -----------------------------
# 1b) Inspect missing values (coded as 'unknown')
# -----------------------------
print("'unknown' counts per column:")
for col in df.select_dtypes(include='object').columns:
    unk = (df[col] == 'unknown').sum()
    if unk > 0:
        print(f"  {col}: {unk} ({unk/len(df)*100:.1f}%)")

In [ ]:
# -----------------------------
# 1c) Encode target and train/val/test split
# -----------------------------
df['y'] = (df['y'] == 'yes').astype(int)

X = df.drop(columns=['y'])
y = df['y']

# 70 / 15 / 15 split (stratified for class balance)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
print(f"Target balance (train): {y_train.mean():.3f} positive")

In [ ]:
# -----------------------------
# 1d) Handle missing values (fit on train only)
# -----------------------------
X_train = X_train.drop(columns=['default'])
X_val = X_val.drop(columns=['default'])
X_test = X_test.drop(columns=['default'])

unknown_cols = ['job', 'marital', 'education', 'housing', 'loan']

modes = {}
for col in unknown_cols:
    modes[col] = X_train[X_train[col] != 'unknown'][col].mode()[0]
    print(f"  {col}: impute 'unknown' -> '{modes[col]}'")

for col in unknown_cols:
    X_train[col] = X_train[col].replace('unknown', modes[col])
    X_val[col] = X_val[col].replace('unknown', modes[col])
    X_test[col] = X_test[col].replace('unknown', modes[col])

In [ ]:
# -----------------------------
# 1e) One-hot encode categorical variables (fit on train only)
# -----------------------------
categorical_cols = ['job', 'marital', 'education', 'housing', 'loan',
                    'contact', 'month', 'day_of_week', 'poutcome']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[categorical_cols])

def encode_and_merge(X, ohe, cat_cols):
    encoded = pd.DataFrame(
        ohe.transform(X[cat_cols]),
        columns=ohe.get_feature_names_out(cat_cols),
        index=X.index
    )
    return pd.concat([X.drop(columns=cat_cols), encoded], axis=1)

X_train = encode_and_merge(X_train, ohe, categorical_cols)
X_val = encode_and_merge(X_val, ohe, categorical_cols)
X_test = encode_and_merge(X_test, ohe, categorical_cols)

print("Transformed shapes:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# -----------------------------
# 1f) Feature scaling (fit on train only)
# -----------------------------
continuous_cols = ['age', 'duration', 'campaign', 'pdays', 'previous',
                   'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
                   'euribor3m', 'nr.employed']

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols])

X_train[continuous_cols] = scaler.transform(X_train[continuous_cols])
X_val[continuous_cols] = scaler.transform(X_val[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

### Why is feature scaling necessary for MLP but not for GBDT?

**MLP (Neural Networks)** use gradient descent to update weights. If features have vastly different scales (e.g., `age` 0–100 vs `nr.employed` 4900–5300), gradients will be dominated by large-scale features, causing slow or unstable convergence. Activation functions like sigmoid also saturate with large inputs, killing the gradient. Standardization ensures all features contribute equally during training.

**Tree-based models (XGBoost)** make binary split decisions like "is euribor3m > 1.5?" The threshold adapts to whatever scale the feature is on. Trees only care about the rank ordering of values, not their magnitude, so scaling has zero effect on performance.

# 2. Gradient Boosted Tree (GBDT)

In [ ]:
# -----------------------------
# 2a) Train XGBoost with different learning rates
# -----------------------------
results = {}

for lr in [0.01, 0.1, 0.3]:
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        learning_rate=lr,
        n_estimators=500,
        max_depth=6,
        subsample=0.9,
        reg_alpha=0.0,
        reg_lambda=1.0,
        early_stopping_rounds=20,
        random_state=42,
        n_jobs=-1
    )
    start = time.time()
    model.fit(X_train, y_train,
              eval_set=[(X_train, y_train), (X_val, y_val)],
              verbose=False)
    train_time = time.time() - start

    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results[f"XGB (lr={lr})"] = {
        'model': model, 'y_pred': y_pred, 'proba': proba, 'time': train_time
    }

    print(f"\n=== XGB (lr={lr}) ===")
    print(f"Stopped at round {model.best_iteration} | Train time: {train_time:.1f}s")
    print(classification_report(y_test, y_pred, target_names=['No', 'Yes'], digits=3))

### Training vs. Validation Loss

In [ ]:
# -----------------------------
# 2b) Visualize: training vs validation loss
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, lr in zip(axes, [0.01, 0.1, 0.3]):
    model = results[f"XGB (lr={lr})"]['model']
    evals = model.evals_result()
    ax.plot(evals['validation_0']['logloss'], label='Train')
    ax.plot(evals['validation_1']['logloss'], label='Validation')
    ax.set_title(f'lr={lr} (stopped at {model.best_iteration})')
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel('Log Loss')
    ax.legend()

plt.suptitle('XGBoost: Training vs Validation Loss', fontsize=14)
plt.tight_layout()
plt.show()

### Feature Importance

In [ ]:
# -----------------------------
# 2c) Visualize: feature importance (using xgb.plot_importance)
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 6))
xgb.plot_importance(results["XGB (lr=0.1)"]['model'], max_num_features=20, ax=ax)
plt.title('XGBoost Feature Importance (plot_importance)')
plt.tight_layout()
plt.show()

### Effect of Learning Rate

In [ ]:
# -----------------------------
# 2d) Compare learning rates
# -----------------------------
print("=" * 80)
print("Effect of Learning Rate on XGBoost")
print("=" * 80)
for name in ["XGB (lr=0.01)", "XGB (lr=0.1)", "XGB (lr=0.3)"]:
    res = results[name]
    report(name, y_test, res['y_pred'], proba=res['proba'])

# Visualize learning rate comparison
lr_names = ["XGB (lr=0.01)", "XGB (lr=0.1)", "XGB (lr=0.3)"]
lr_f1s = [f1_score(y_test, results[n]['y_pred']) for n in lr_names]

plt.figure(figsize=(7, 4))
plt.bar(['0.01', '0.1', '0.3'], lr_f1s, color=['#4C72B0', '#55A868', '#C44E52'])
plt.xlabel('Learning Rate')
plt.ylabel('F1 Score')
plt.title('Effect of Learning Rate on XGBoost Performance')
plt.tight_layout()
plt.show()

### Explore Other Hyperparameters

In [ ]:
# -----------------------------
# 2e) Effect of n_estimators
# -----------------------------
for n in [50, 100, 300, 500]:
    model = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        learning_rate=0.1, n_estimators=n, max_depth=6,
        subsample=0.9, reg_lambda=1.0, random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results[f"XGB (n_est={n})"] = {'y_pred': y_pred, 'proba': proba}

print("=" * 80)
print("Effect of n_estimators")
print("=" * 80)
for n in [50, 100, 300, 500]:
    res = results[f"XGB (n_est={n})"]
    report(f"XGB (n_est={n})", y_test, res['y_pred'], proba=res['proba'])

In [ ]:
# -----------------------------
# 2f) Effect of max_depth
# -----------------------------
for depth in [3, 6, 10]:
    model = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        learning_rate=0.1, n_estimators=300, max_depth=depth,
        subsample=0.9, reg_lambda=1.0, random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results[f"XGB (depth={depth})"] = {'y_pred': y_pred, 'proba': proba}

print("=" * 80)
print("Effect of max_depth")
print("=" * 80)
for depth in [3, 6, 10]:
    res = results[f"XGB (depth={depth})"]
    report(f"XGB (depth={depth})", y_test, res['y_pred'], proba=res['proba'])

In [ ]:
# -----------------------------
# 2g) Effect of subsample
# -----------------------------
for ss in [0.5, 0.7, 0.9, 1.0]:
    model = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        learning_rate=0.1, n_estimators=300, max_depth=6,
        subsample=ss, reg_lambda=1.0, random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results[f"XGB (subsample={ss})"] = {'y_pred': y_pred, 'proba': proba}

print("=" * 80)
print("Effect of subsample")
print("=" * 80)
for ss in [0.5, 0.7, 0.9, 1.0]:
    res = results[f"XGB (subsample={ss})"]
    report(f"XGB (subsample={ss})", y_test, res['y_pred'], proba=res['proba'])

In [ ]:
# -----------------------------
# 2h) Effect of reg_alpha and reg_lambda
# -----------------------------
for alpha, lam in [(0, 0), (0.1, 1.0), (1.0, 5.0)]:
    model = xgb.XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        learning_rate=0.1, n_estimators=300, max_depth=6,
        subsample=0.9, reg_alpha=alpha, reg_lambda=lam,
        random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results[f"XGB (a={alpha},l={lam})"] = {'y_pred': y_pred, 'proba': proba}

print("=" * 80)
print("Effect of Regularization (reg_alpha, reg_lambda)")
print("=" * 80)
for alpha, lam in [(0, 0), (0.1, 1.0), (1.0, 5.0)]:
    res = results[f"XGB (a={alpha},l={lam})"]
    report(f"XGB (a={alpha},l={lam})", y_test, res['y_pred'], proba=res['proba'])

### Hyperparameter Search

In [ ]:
# -----------------------------
# 2i) RandomizedSearchCV for best hyperparameters
# -----------------------------
param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'max_depth': [3, 6, 8, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [0, 1.0, 5.0],
    'n_estimators': [100, 300, 500]
}

xgb_search = RandomizedSearchCV(
    xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1
)
xgb_search.fit(X_train, y_train)
print("Best GBDT params:", xgb_search.best_params_)
print("Best CV F1:", f"{xgb_search.best_score_:.4f}")

# 3. Multi-Layer Perceptron (MLP)

In [ ]:
# -----------------------------
# 3a) Train MLP with different architectures
# -----------------------------
for layers in [(64,), (128, 64), (256, 128, 64)]:
    mlp = MLPClassifier(
        hidden_layer_sizes=layers,
        activation='relu',
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42
    )
    start = time.time()
    mlp.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = mlp.predict(X_test)
    proba = mlp.predict_proba(X_test)[:,1]
    results[f"MLP {layers}"] = {
        'model': mlp, 'y_pred': y_pred, 'proba': proba, 'time': train_time
    }

print("=" * 80)
print("Effect of Network Depth/Width")
print("=" * 80)
for layers in [(64,), (128, 64), (256, 128, 64)]:
    res = results[f"MLP {layers}"]
    report(f"MLP {layers}", y_test, res['y_pred'], proba=res['proba'])

In [ ]:
# -----------------------------
# 3b) Activation function comparison
# -----------------------------
for act in ['relu', 'tanh']:
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation=act,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42
    )
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    proba = mlp.predict_proba(X_test)[:,1]
    results[f"MLP relu vs tanh ({act})"] = {
        'model': mlp, 'y_pred': y_pred, 'proba': proba
    }

print("=" * 80)
print("Effect of Activation Function")
print("=" * 80)
for act in ['relu', 'tanh']:
    res = results[f"MLP relu vs tanh ({act})"]
    report(f"MLP ({act})", y_test, res['y_pred'], proba=res['proba'])

In [ ]:
# -----------------------------
# 3c) Learning rate comparison
# -----------------------------
for lr in [0.001, 0.01, 0.1]:
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        learning_rate_init=lr,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42
    )
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    proba = mlp.predict_proba(X_test)[:,1]
    results[f"MLP (lr_init={lr})"] = {
        'model': mlp, 'y_pred': y_pred, 'proba': proba
    }

print("=" * 80)
print("Effect of Learning Rate")
print("=" * 80)
for lr in [0.001, 0.01, 0.1]:
    res = results[f"MLP (lr_init={lr})"]
    report(f"MLP (lr_init={lr})", y_test, res['y_pred'], proba=res['proba'])

In [ ]:
# -----------------------------
# 3d) Effect of max_iter
# -----------------------------
for mi in [50, 100, 200, 300]:
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        learning_rate_init=0.001,
        max_iter=mi,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42
    )
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)
    proba = mlp.predict_proba(X_test)[:,1]
    results[f"MLP (max_iter={mi})"] = {
        'model': mlp, 'y_pred': y_pred, 'proba': proba
    }

print("=" * 80)
print("Effect of max_iter")
print("=" * 80)
for mi in [50, 100, 200, 300]:
    res = results[f"MLP (max_iter={mi})"]
    report(f"MLP (max_iter={mi})", y_test, res['y_pred'], proba=res['proba'])

### Training Loss Curves

In [ ]:
# -----------------------------
# 3e) Visualize: training loss curves by architecture
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, layers in zip(axes, [(64,), (128, 64), (256, 128, 64)]):
    mlp = results[f"MLP {layers}"]['model']
    ax.plot(mlp.loss_curve_, label='Train Loss')
    ax.set_title(f'MLP {layers}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend()

plt.suptitle('MLP: Training Loss Curves by Architecture', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# 3f) Visualize: training loss curves by learning rate
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lr in zip(axes, [0.001, 0.01, 0.1]):
    mlp = results[f"MLP (lr_init={lr})"]['model']
    ax.plot(mlp.loss_curve_, label='Train Loss')
    ax.set_title(f'lr_init={lr}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend()

plt.suptitle('MLP: Training Loss Curves by Learning Rate', fontsize=14)
plt.tight_layout()
plt.show()

### Effect of Network Depth/Width on Validation Performance

In [ ]:
# -----------------------------
# 3g) Visualize: depth/width vs performance
# -----------------------------
layer_configs = [(64,), (128, 64), (256, 128, 64)]
val_f1s = []
for layers in layer_configs:
    res = results[f"MLP {layers}"]
    val_f1s.append(f1_score(y_test, res['y_pred']))

plt.figure(figsize=(7, 4))
plt.bar([str(l) for l in layer_configs], val_f1s, color=['#4C72B0', '#55A868', '#C44E52'])
plt.xlabel('Hidden Layer Sizes')
plt.ylabel('F1 Score')
plt.title('Effect of Network Depth/Width on Performance')
plt.tight_layout()
plt.show()

# 4. GBDT vs MLP Comparison

In [ ]:
# -----------------------------
# 4a) Side-by-side report
# -----------------------------
best_xgb_name = "XGB (lr=0.1)"
best_mlp_name = "MLP (128, 64)"

print("=" * 80)
print("GBDT vs MLP: Final Comparison")
print("=" * 80)
report(best_xgb_name, y_test, results[best_xgb_name]['y_pred'], proba=results[best_xgb_name]['proba'])
report(best_mlp_name, y_test, results[best_mlp_name]['y_pred'], proba=results[best_mlp_name]['proba'])

print("\nDetailed classification reports:")
for name in [best_xgb_name, best_mlp_name]:
    print(f"\n=== {name} ===")
    print(classification_report(y_test, results[name]['y_pred'], target_names=['No', 'Yes'], digits=3))

In [ ]:
# -----------------------------
# 4b) Cross-validation (robustness check)
# -----------------------------
xgb_cv_model = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='logloss',
    learning_rate=0.1, n_estimators=300, max_depth=6,
    subsample=0.9, reg_lambda=1.0, random_state=42, n_jobs=-1
)
xgb_cv = cross_val_score(xgb_cv_model, X_train, y_train, cv=5, scoring='f1')
print(f"XGB 5-fold CV F1: {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

mlp_cv_model = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu',
    learning_rate_init=0.001, max_iter=300, random_state=42
)
mlp_cv = cross_val_score(mlp_cv_model, X_train, y_train, cv=5, scoring='f1')
print(f"MLP 5-fold CV F1: {mlp_cv.mean():.4f} ± {mlp_cv.std():.4f}")

In [ ]:
# -----------------------------
# 4c) Summary comparison table
# -----------------------------
rows = []
for name in [best_xgb_name, best_mlp_name]:
    res = results[name]
    rows.append({
        'Model': name,
        'Accuracy': f"{accuracy_score(y_test, res['y_pred']):.4f}",
        'Precision': f"{precision_score(y_test, res['y_pred']):.4f}",
        'Recall': f"{recall_score(y_test, res['y_pred']):.4f}",
        'F1': f"{f1_score(y_test, res['y_pred']):.4f}",
        'AUC-PR': f"{average_precision_score(y_test, res['proba']):.4f}",
        'Train Time (s)': f"{res['time']:.2f}"
    })

comparison_df = pd.DataFrame(rows).set_index('Model')
print("\nSummary comparison:\n")
print(comparison_df.to_string())

In [ ]:
# -----------------------------
# 4d) ROC & PR curves (class imbalance aware)
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC (left)
ax = axes[0]
for name in [best_xgb_name, best_mlp_name]:
    fpr, tpr, _ = roc_curve(y_test, results[name]['proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()

# PR (right) — more informative for imbalanced data
ax = axes[1]
for name in [best_xgb_name, best_mlp_name]:
    prec, rec, _ = precision_recall_curve(y_test, results[name]['proba'])
    ap = average_precision_score(y_test, results[name]['proba'])
    ax.plot(rec, prec, label=f"{name} (AP={ap:.3f})")
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# 4e) Confusion matrices
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, [best_xgb_name, best_mlp_name]):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f"{name} – Confusion Matrix")

plt.suptitle('GBDT vs MLP: Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

### Discussion

**When to prefer GBDT over MLP?** GBDT is better suited for tabular data — it trains faster, requires less preprocessing, and typically achieves strong performance with minimal tuning. MLP is preferable when data has complex nonlinear interactions, very large sample sizes, or when integrating with other neural network components.

**Interpretability:** GBDT provides feature importance scores (gain, weight, cover) showing which features drive predictions. MLP is a black box — weights spread across multiple hidden layers are not directly interpretable without additional tools like SHAP or LIME.

**Categorical features and missing values:** XGBoost handles missing values natively by learning optimal split directions during training. MLP requires all features to be numeric and complete — categoricals must be one-hot encoded and missing values must be imputed before training.

**Hyperparameter sensitivity:** MLP is more sensitive — learning rate, architecture size, and feature scaling can make the difference between a strong model and a broken one. GBDT is more forgiving and often performs competitively with default parameters.